# 05 - Single Turn Model: comparing embeddings

The purpose of this notebook is to train one FFNN per embedding (GPT-2, Nomic, Qwen3) on single-turn data, select the best by validation PR-AUC, then check whether that model transfers to multi-turn conversations it never saw.  

#### Load Dependencies and set directory structure

In [47]:
# Load dependencies and files

# Set dependencies
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    classification_report, confusion_matrix,
    average_precision_score, roc_auc_score
)
from pyprojroot import here

# Project path anchors
REPO_ROOT = here()
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"
MODEL_DIR = REPO_ROOT / "data" / "models"

#### Load Embeddings and Labels

In [ ]:
# metadata (row-aligned with every .npy)
meta = pd.read_parquet(PROCESSED_DATA_DIR / "singleturn_all.parquet")
mt_meta = pd.read_parquet(PROCESSED_DATA_DIR / "multiturn_test_all.parquet")

EMB_NAMES = ("gpt2", "qwen3", "nomic")
singleturn_emb = {n: np.load(PROCESSED_DATA_DIR / f"singleturn_emb_{n}.npy") for n in EMB_NAMES}
multiturn_emb = {n: np.load(PROCESSED_DATA_DIR / f"multiturn_test_emb_{n}.npy") for n in EMB_NAMES}

# Alignment checks
for n in EMB_NAMES:
    assert singleturn_emb[n].shape[0] == len(meta)
    assert multiturn_emb[n].shape[0] == len(mt_meta)
    assert np.isfinite(singleturn_emb[n]).all() and np.isfinite(multiturn_emb[n]).all()
    print(f"{n:>5}: singleturn {singleturn_emb[n].shape}, multiturn {multiturn_emb[n].shape}")


In [ ]:
def get_split(emb_name, split):
    mask = (meta["split"] == split).to_numpy()
    return singleturn_emb[emb_name][mask], meta.loc[mask, "harm"].to_numpy()

y_mt = mt_meta["harm"].to_numpy()
